# Seedance en Colab — 12 workflows golden

Notebook para correr los **12 workflows de Seedance** de la libreria golden.

**Este notebook NO necesita GPU.** 10 de los 12 workflows son 100% nodos de API
(ByteDance / Gemini / OpenAI): el video se genera en los servidores de ellos, esta
maquina solo manda la imagen y baja el mp4. Podes usar un entorno de ejecucion
**CPU** y no gastar tu cuota de GPU.

Las dos excepciones estan marcadas en el chequeo de la ultima celda.

**Se paga con creditos de comfy.org**, no con Colab. El login se hace en la interfaz
web de ComfyUI (por el tunel), NO en este notebook: aca no se escribe ninguna clave.

> Ficha completa de cada workflow: `seedance/REGISTRY.md` en este mismo repo.


## 0) Entorno — que tenemos

No falla si no hay GPU: la mayoria de los workflows no la necesitan.


In [ ]:
import subprocess, shutil, sys

r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
HAY_GPU = (r.returncode == 0)
if HAY_GPU:
    print('GPU detectada:')
    print(r.stdout.split('\n')[8] if len(r.stdout.split('\n')) > 8 else r.stdout[:300])
else:
    print('Sin GPU en este entorno — ESTA BIEN.')
    print('10 de los 12 workflows son 100% nodos de API y no la usan.')
    print('Solo la necesitan: seedance20_depthmotion (DepthAnythingV2) y')
    print('product_ad_video_seedance (MiniMax H3 local, que igual no entra en una T4).')

print()
print(f'Python  : {sys.version.split()[0]}')
print(f'Disco   : {shutil.disk_usage("/content").free/1e9:.1f} GB libres en la VM')


## 1) Google Drive (opcional pero recomendado)

La VM de Colab se borra al cerrar la sesion. Si montas Drive, los videos generados
quedan guardados; si no, se pierden al desconectarte.


In [ ]:
USAR_DRIVE = True   # False = todo efimero, se pierde al cerrar la sesion

import os
SALIDAS = '/content/ComfyUI/output'
if USAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SALIDAS = '/content/drive/MyDrive/ComfyUI-Leo/seedance-output'
    os.makedirs(SALIDAS, exist_ok=True)
    print('Los videos se van a guardar en:', SALIDAS)
else:
    print('Sin Drive: las salidas quedan en la VM y se pierden al cerrar.')


## 2) Clonar ComfyUI

Se clona **pineado en `v0.34.3`**, que es la version que la libreria declara como
last-known-good. Es la primera que trae `ByteDance2ReferenceNodeV2`, el nodo que
necesita el workflow de **Video Extend**.


In [ ]:
# Custom nodes: solo hacen falta para seedance20_depthmotion. Dejalo en False
# salvo que vayas a correr ese workflow puntual.
INSTALAR_NODOS_DEPTH = False

COMFY_TAG = 'v0.34.3'

import os
%cd /content
if not os.path.isdir('/content/ComfyUI'):
    !git clone --depth 1 --branch {COMFY_TAG} https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt

if INSTALAR_NODOS_DEPTH:
    CN = '/content/ComfyUI/custom_nodes'
    for repo in ['https://github.com/Fannovel16/comfyui_controlnet_aux',
                 'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite']:
        d = os.path.join(CN, repo.rstrip('/').split('/')[-1])
        if not os.path.isdir(d):
            !git clone -q {repo} {d}
        req = os.path.join(d, 'requirements.txt')
        if os.path.isfile(req):
            !pip install -q -r {req}
    print('Custom nodes de depth instalados.')
else:
    print('Sin custom nodes (no hacen falta para 11 de los 12 workflows).')

import comfyui_version
print('ComfyUI', comfyui_version.__version__)


## 3) Cargar los 12 workflows golden

Se copian a `user/default/workflows/`, asi te aparecen directo en la barra lateral
de la interfaz de ComfyUI. Los originales quedan intactos en el repo.


In [ ]:
import os, shutil, glob
%cd /content

REPO = '/content/comfy-colab-leo'
if not os.path.isdir(REPO):
    !git clone -q https://github.com/nadikka/comfy-colab-leo {REPO}
else:
    !git -C {REPO} pull -q

DST = '/content/ComfyUI/user/default/workflows'
os.makedirs(DST, exist_ok=True)
golden = sorted(glob.glob(os.path.join(REPO, 'seedance', 'golden', '*.json')))
for g in golden:
    shutil.copy2(g, os.path.join(DST, os.path.basename(g)))
print(f'{len(golden)} workflows golden cargados en la barra lateral de ComfyUI:')
for g in golden:
    print('  -', os.path.basename(g))


## 4) Creditos de comfy.org — se hace en el navegador, no aca

Los nodos de ByteDance / Gemini / OpenAI se cobran con **creditos de comfy.org**.

Cuando la ultima celda te de la URL del tunel:

1. Abri esa URL.
2. En ComfyUI, arriba a la derecha, entra a tu cuenta de **comfy.org**.
3. Los nodos de API quedan habilitados con tus creditos.

**No pongas ninguna clave ni credencial en este notebook.** Es un repo publico:
cualquier cosa que escribas aca queda expuesta y en el historial de git.

> Antes de tirar el primer render caro: mira el indice de `seedance/REGISTRY.md`.
> Los `cost_tier` son relativos. Arranca por `seedance25_r2v` a 720p (tier BAJO)
> y recien cuando funcione escala a 1080p/4k.


## 5) Levantar ComfyUI + tunel

Al arrancar imprime un **chequeo de disponibilidad**: cuales de los 12 workflows
tienen todos sus nodos presentes en esta instalacion y cuales no.

> Si tenes el secreto `CF_TUNNEL_CRED` cargado, usa la URL fija
> `comfy.leoblumfeld.com`. **Ojo:** no puede estar corriendo al mismo tiempo que
> `ComfyUI_Leo.ipynb` — los dos pelean por el mismo tunel.


In [ ]:
import os, json, glob, re, threading, time, socket, subprocess, urllib.request
%cd /content/ComfyUI

if not os.path.isfile('/content/ComfyUI/main.py'):
    raise RuntimeError("Falta /content/ComfyUI/main.py — corre primero la celda 2.")

UUID_RE = re.compile(r'^[0-9a-f]{8}-[0-9a-f]{4}-', re.I)
IGNORAR = {'MarkdownNote', 'Note', 'Reroute'}

def tipos_de(path):
    """Extrae los tipos de nodo de un workflow, incluyendo los de sus subgrafos."""
    d = json.load(open(path, encoding='utf-8'))
    t = set()
    def walk(ns):
        for nd in ns:
            ty = nd.get('type')
            if ty and not UUID_RE.match(ty) and ty not in IGNORAR:
                t.add(ty)
    walk(d.get('nodes', []))
    for sg in (d.get('definitions') or {}).get('subgraphs') or []:
        walk(sg.get('nodes', []))
    return t

def wait_8188():
    while socket.socket().connect_ex(('127.0.0.1', 8188)) != 0:
        time.sleep(1)

def chequeo():
    wait_8188()
    time.sleep(4)
    try:
        disponibles = set(json.load(urllib.request.urlopen(
            'http://127.0.0.1:8188/object_info', timeout=60)).keys())
    except Exception as e:
        print('No se pudo leer /object_info:', e)
        return
    print('\n' + '=' * 62)
    print('  CHEQUEO DE DISPONIBILIDAD — 12 workflows golden')
    print('=' * 62)
    ok = 0
    for w in sorted(glob.glob('/content/ComfyUI/user/default/workflows/*.json')):
        faltan = sorted(tipos_de(w) - disponibles)
        nom = os.path.basename(w).replace('.json', '')
        if faltan:
            print(f'  [NO]  {nom}\n          falta: {", ".join(faltan)}')
        else:
            ok += 1
            print(f'  [OK]  {nom}')
    print('=' * 62)
    print(f'  {ok}/12 listos para correr en esta instalacion.')
    print('=' * 62 + '\n')

threading.Thread(target=chequeo, daemon=True).start()

# ---------------- tunel ----------------
CRED = None
try:
    from google.colab import userdata
    CRED = userdata.get('CF_TUNNEL_CRED')
except Exception as e:
    print('Sin secreto CF_TUNNEL_CRED ->', e)

if CRED:
    open('/content/comfy-leo.json', 'w').write(CRED)
    if not os.path.isfile('/usr/local/bin/cloudflared'):
        os.system('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/'
                  'cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && '
                  'chmod +x /usr/local/bin/cloudflared')
    def tunnel():
        wait_8188()
        print('\n\n' + '=' * 48)
        print('  URL FIJA:  https://comfy.leoblumfeld.com')
        print('=' * 48 + '\n')
        # --protocol http2: Colab estrangula QUIC/UDP y el tunel se corta en loop.
        subprocess.run(['cloudflared', 'tunnel', '--no-autoupdate',
                        '--protocol', 'http2', '--retries', '10',
                        '--grace-period', '30s',
                        '--url', 'http://127.0.0.1:8188',
                        'run', '--credentials-file', '/content/comfy-leo.json',
                        'comfy-leo'])
    threading.Thread(target=tunnel, daemon=True).start()
else:
    os.system('pip install -q pycloudflared')
    def tunnel():
        wait_8188()
        from pycloudflared import try_cloudflare
        url = try_cloudflare(port=8188).tunnel
        print('\n\n' + '=' * 48)
        print('  URL PUBLICA (random):')
        print('  ' + url)
        print('=' * 48 + '\n')
    threading.Thread(target=tunnel, daemon=True).start()

ARGS = f'--output-directory "{SALIDAS}"' if 'SALIDAS' in dir() else ''
!python main.py --dont-print-server {ARGS}
